In [1]:
import numpy as np
import pandas as pd

# ── Configuration: header rows ──────────────────────────────────────────────
HEADER_ROW_GALLIKER = 0
HEADER_ROW_SPOT     = 0

# ── Data: Load profiles (Galliker) ──────────────────────────────────────────
galliker_path = (
    r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team'
    r'\Caspar\Model_HEMS\hems_resopt\data\01_raw\260707_Lastgänge_galliker.xlsx'
)
df_galliker = pd.read_excel(galliker_path, header=HEADER_ROW_GALLIKER)
df_galliker = df_galliker.drop(index=0)

# ── Price Data (Swiss day-ahead spot) ───────────────────────────────────────
spot_ch_path = (
    r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team'
    r'\Caspar\Model_HEMS\hems_resopt\data\01_raw\spot_ch_25.xlsx'
)
df_spot = pd.read_excel(spot_ch_path, header=HEADER_ROW_SPOT)



# ── Time Index ───────────────────────────────────────────────────────────────
# Built from the actual length of the load-profile data (15-min resolution).
start_time = pd.Timestamp('2025-01-01', tz='Europe/Zurich')
idx = pd.date_range(
    start=start_time,
    freq='15min',
    periods=len(df_galliker),
    tz='Europe/Zurich',
)

df_galliker.index = idx
df_galliker = df_galliker.drop(columns='Unnamed: 0')
df_galliker = df_galliker.astype(float)
df_galliker = df_galliker * 4 / 1000 # from kWh to MW

spot_column = 'Spot CH [Euro/MWh]'
df_spot = df_spot.iloc[:len(df_galliker)]
df_spot.index = idx
spot_price_series = df_spot['Spot CH [Euro/MWh]']

df_galliker['Total_San_Antonino'] = df_galliker['Bezug San Antonino'] - df_galliker['Überschuss San Antonino']
df_galliker['Total_Schachen'] = df_galliker['Bezug Schachen'] - df_galliker['Überschuss Schachen']

p_max_antinino = df_galliker['Bezug San Antonino'].max()
p_max_schachen = df_galliker['Bezug Schachen'].max()

# Run Simulation
## Run entire time horizon at once

In [2]:
# import pyomo.environ as pyo
# from hems_resopt.components.grid import GridPeakShave
# from res_opt_core import EnergyModel, Battery, AuctionMarket, plot_battery_operation, Grid, Load, Pv
# from res_opt_core.core.components.energy_balance import GridLimits
# from res_opt_core.valuator.components.grid import GridLimitsWithFees
# from hems_resopt.components.ev import EV
# from pyomo.contrib.solver.solvers.highs import Highs

# # Change the solver attributes so that solving doesnt take too long
# custom_solver = Highs()
# # custom_solver.config.time_limit = 300.0
# custom_solver.config.rel_gap = 0.05          # the gap allowed to the optimal solution
# # custom_solver.config.solver_options = {
# #     "presolve": "on",
# #     "mip_heuristic_effort": 0.2,
# #     "output_flag": True,     # ensure HiGHS actually prints its log
# #     "log_to_console": True,  # some HiGHS builds use this name instead/additionally
# # }

# # ── Simulation Configuration ───────────────────────────────────────────────────
# p_max_Netz = p_max_antinino
# kosten_lastspitze = 7190        # CHF/MW
# arbeitspreis_netz_bezug = 42.4    # CHF/MWh
# arbeitspreis_netz_einspeisung = 0

# Anschlussleistung_SanAntonino = 1.6  # MW
# cos_phi = 0.86
# Anschlussleistung_Schachen = 0.935 * cos_phi  # MW

# output_dir = r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed'

# # ── Model ─────────────────────────────────────────────────────────────────────
# model = EnergyModel(
#     num_steps=len(df_galliker.index),
#     slot_length="15min",
#     solver=custom_solver,
#     timestamp=df_galliker.index,
# )


# # ── Load ───────────────────────────────
# Building = Load(
#     name='San_Antonino',
#     load_curve=df_galliker['Bezug San Antonino'],
#     lost_load_allowed=False, # Kann Load ignoriert werden -> nicht Erfüllung der Menge die bezogen werden sollte
#     value_of_lost_load=1000.0 
# )
# model.add_component(Building)

# # ── PV ───────────────────────────────
# pv = Pv(
#     name='PV_San_Antonino',
#     power_max=df_galliker['Überschuss San Antonino'],
#     power_min=0,
#     # allow_spill=True, # Man kann curtailen (last kappen)
#     # cost_spill=2 # curtailment price
# )
# model.add_component(pv)

# # ── BESS ───────────────────────────────
# bess = Battery(
#     name='bess_san_antonino',
#     power_nominal=5,
#     energy_capacity=15,
#     eta=0.85 # efficiency
# )
# model.add_component(bess)

# # ── Market ──────────────────────────────────────────────────────────────────
# spot_ch = AuctionMarket(
#     name='spot_ch',
#     price_curve=spot_price_series,
#     market_time_unit='hour' # definiert die Auflösung des Marktes = muss produkt für 1h kaufen
# )
# model.add_component(spot_ch)

# # ── Grid  ─────────────────────
# grid = GridPeakShave(
#     name='Grid_with_peakshave',
#     assets=[bess, pv, Building],
#     markets=[spot_ch],
#     peak_power_price=kosten_lastspitze,
#     time_horizon_peak='month_daylight_saving',
#     power_max=Anschlussleistung_SanAntonino,
#     power_min=-Anschlussleistung_SanAntonino,
#     fee_export=arbeitspreis_netz_einspeisung,
#     fee_import=arbeitspreis_netz_bezug,
# )
# model.add_component(grid)

# # grid = GridLimitsWithFees(
# #     inbound_components=[],
# #     name = 'Grid_with_Limits',
# #     power_max = Anschlussleistung_SanAntonino,
# #     power_min = -Anschlussleistung_SanAntonino,
# #     fee_export=0,
# #     fee_import=100,
# # )
# # model.add_component(grid)

# # ── Run ───────────────────────────────────────────────────────────────────────
# model.build_and_run(silent=False)  # solver output visible

# df_results = model.results.timeseries_to_pandas()
# df_results.to_csv(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed\galliker_san_antonino.csv')

## Run every Month individually


In [ ]:
import pyomo.environ as pyo
from hems_resopt.components.grid import GridPeakShave
from res_opt_core import EnergyModel, Battery, AuctionMarket, plot_battery_operation, Grid, Load, Pv
from res_opt_core.core.components.energy_balance import GridLimits
from res_opt_core.valuator.components.grid import GridLimitsWithFees
from hems_resopt.components.ev import EV
from pyomo.contrib.solver.solvers.highs import Highs
import os

# Change the solver attributes so that solving doesnt take too long
custom_solver = Highs()
# custom_solver.config.time_limit = 330.0 #Currently no time limit gap and only gap to optimal solution as finishing line
custom_solver.config.rel_gap = 0.065          # the gap allowed to the optimal solution
custom_solver.config.solver_options = {
    "presolve": "on",
    "mip_heuristic_effort": 0.2,
}

# ── Simulation Configuration ───────────────────────────────────────────────────
p_max_Netz = p_max_antinino
kosten_lastspitze = 7190        # CHF/MW
arbeitspreis_netz_bezug = 42.4    # CHF/MWh
arbeitspreis_netz_einspeisung = 0

Anschlussleistung_SanAntonino = 1.6  # MW
cos_phi = 0.86
Anschlussleistung_Schachen = 0.935 * cos_phi  # MW

output_dir = r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed'

## Run 2025
df_galliker = df_galliker.loc['2025']
spot_price_series = spot_price_series.loc['2025']

# ── Build monthly input dictionaries ────────────────────────────────────────────
months = df_galliker.index.to_series().dt.to_period('M').unique()
months = sorted(months)

galliker_by_month = {}
spot_by_month = {}

for month in months:
    month_mask = df_galliker.index.to_period('M') == month
    galliker_by_month[month] = df_galliker.loc[month_mask].copy()
    spot_by_month[month] = spot_price_series.loc[month_mask].copy()

print(f"ℹ️  Split data into {len(months)} monthly chunks: {[str(m) for m in months]}")

# ── Run the simulation once per month ──────────────────────────────────────────
results_per_month = {}

for month in months:
    csv_path = os.path.join(output_dir, f'galliker_schachen_{month}.csv')

    if os.path.exists(csv_path):
        print(f"⏭️  Skipping month {month} — CSV already exists at {csv_path}")
        continue

    print(f"\n{'='*60}")
    print(f"  Running simulation for month: {month}")
    print(f"{'='*60}")

    try:
        df_galliker_m = galliker_by_month[month]
        spot_m = spot_by_month[month]
        idx_m = df_galliker_m.index


        # ── Model ─────────────────────────────────────────────────────────────
        model = EnergyModel(
            num_steps=len(idx_m),
            slot_length="15min",
            solver=custom_solver,
            timestamp=idx_m,
        )

        # ── Load ──────────────────────────────────────────────────────────────
        Building = Load(
            name='Schachen',
            load_curve=df_galliker_m['Bezug Schachen'],
            lost_load_allowed=False,
            # value_of_lost_load=1000.0,
        )
        model.add_component(Building)

        # ── PV ────────────────────────────────────────────────────────────────
        pv = Pv(
            name='PV_Schachen',
            power_max=df_galliker_m['Überschuss Schachen'],
            # power_min=0,
            allow_spill=False,
            # cost_spill=2,
        )
        model.add_component(pv)

        # ── BESS ──────────────────────────────────────────────────────────────
        bess = Battery(
            name='bess_schachen',
            power_nominal=5,
            energy_capacity=15,
            eta=0.85,
        )
        model.add_component(bess)

        # ── Market ────────────────────────────────────────────────────────────
        spot_ch = AuctionMarket(
            name='spot_ch',
            price_curve=spot_m,
            market_time_unit='hour',
        )
        model.add_component(spot_ch)

        # ── Grid ──────────────────────────────────────────────────────────────
        # Behind-the-meter Case
        grid = GridPeakShave(
            name='Grid_with_peakshave',
            assets=[bess, pv, Building],
            markets=[spot_ch],
            peak_power_price=kosten_lastspitze,
            time_horizon_peak='month_daylight_saving',
            power_max=Anschlussleistung_Schachen,
            power_min=-Anschlussleistung_Schachen,
            fee_export=arbeitspreis_netz_einspeisung,
            fee_import=arbeitspreis_netz_bezug,
        )

        # In Front-of-the-meter Case -> no grid fees
        # grid = Grid(
        #     name='Grid-no-grid-fees',
        #     assets=[bess, pv, Building],
        #     markets=[spot_ch],
        #     power_max=Anschlussleistung_Schachen,
        #     power_min=-Anschlussleistung_Schachen,
        # )
        model.add_component(grid)

        # ── Run ───────────────────────────────────────────────────────────────
        model.build_and_run(silent=False)

        df_results_m = model.results.timeseries_to_pandas()

        results_per_month[month] = {
            'df_results': df_results_m,
            'df_galliker': df_galliker_m,
            'spot_price': spot_m,
        }

        print(f"✅ Month {month} solved. df_results shape: {df_results_m.shape}")

        df_results_m.to_csv(csv_path)
        print(f"💾 Saved: {csv_path}")

    except Exception as e:
        print(f"⛔ Month {month} FAILED — {type(e).__name__}: {e}")
        results_per_month[month] = {'error': str(e)}
        continue

print("\nAll months processed.")

# ── Final status summary ───────────────────────────────────────────────────────
for month, res in results_per_month.items():
    status = '⛔ FAILED' if 'error' in res else '✅ OK'
    print(f"  {month}: {status}")


C:\Users\ckw-ThCa\AppData\Local\Temp\ipykernel_21228\1484119870.py:36: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = df_galliker.index.to_series().dt.to_period('M').unique()
C:\Users\ckw-ThCa\AppData\Local\Temp\ipykernel_21228\1484119870.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  month_mask = df_galliker.index.to_period('M') == month


ℹ️  Split data into 12 monthly chunks: ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']

  Running simulation for month: 2025-01
Running HiGHS 1.14.0 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
MIP has 145825 rows; 98208 cols; 316942 nonzeros; 11904 integer variables (11904 binary)
Coefficient ranges:
  Matrix  [1e-02, 5e+00]
  Cost    [1e+01, 7e+03]
  Bound   [1e+00, 1e+00]
  RHS     [4e-04, 5e+00]
Presolving model
35710 rows, 21576 cols, 71418 nonzeros 0s
28053 rows, 19873 cols, 69724 nonzeros 0s
25809 rows, 19873 cols, 65232 nonzeros 0s
Presolve reductions: rows 25809(-120016); columns 19873(-78335); nonzeros 65232(-251710) 

Solving MIP model with:
   25809 rows
   19873 cols (5952 binary, 0 integer, 0 implied int., 13921 continuous, 0 domain fixed)
   65232 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibili

In [ ]:
# ── Rebuild results_per_month from saved CSVs, then concatenate ───────────────
import glob

csv_files = sorted(glob.glob(os.path.join(output_dir, 'galliker_schachen_*.csv')))

# Exclude the combined yearly file itself, in case it already exists in the folder
csv_files = [f for f in csv_files if 'galliker_schachen_2025.csv' not in os.path.basename(f)]

print(f"ℹ️  Found {len(csv_files)} monthly CSV files to combine.")

monthly_dfs = {}
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    month_label = filename.replace('galliker_schachen_', '').replace('.csv', '')

    df_month = pd.read_csv(csv_file, index_col=0)
    # Explicitly convert to datetime, coercing any unparseable values to NaT
    # so we can detect and inspect them rather than silently mixing types.
    df_month.index = pd.to_datetime(df_month.index, errors='coerce', utc=True)

    bad_rows = df_month.index.isna().sum()
    if bad_rows > 0:
        print(f"⚠️  {month_label}: {bad_rows} rows had an unparseable index — dropping them.")
        df_month = df_month[df_month.index.notna()]

    monthly_dfs[month_label] = df_month
    print(f"  Loaded {month_label}: shape {df_month.shape}, index dtype: {df_month.index.dtype}")

if monthly_dfs:
    df_results_all_months = pd.concat(monthly_dfs.values(), axis=0).sort_index()
    print(f"\n📊 Combined df_results shape across all months: {df_results_all_months.shape}")

    combined_path = os.path.join(output_dir, 'galliker_schachen_2025.csv')
    df_results_all_months.to_csv(combined_path)
    print(f"💾 Saved combined file: {combined_path}")
else:
    print("⚠️  No monthly CSVs found — nothing to combine.")



ℹ️  Found 7 monthly CSV files to combine.
  Loaded 2026-01: shape (2976, 29), index dtype: datetime64[us, UTC]
  Loaded 2026-02: shape (2688, 29), index dtype: datetime64[us, UTC]
  Loaded 2026-03: shape (2972, 29), index dtype: datetime64[us, UTC]
  Loaded 2026-04: shape (2880, 29), index dtype: datetime64[us, UTC]
  Loaded 2026-05: shape (2976, 29), index dtype: datetime64[us, UTC]
  Loaded 2026-06: shape (2880, 29), index dtype: datetime64[us, UTC]
  Loaded 2026-07: shape (4, 29), index dtype: datetime64[us, UTC]

📊 Combined df_results shape across all months: (17376, 29)
💾 Saved combined file: C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed\galliker_schachen_2026.csv
